## Task 1: Data Ingestion & Structural Audit

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("saas_telemetry_raw.csv")

print("Shape:", df.shape)
df.head()

Shape: (300, 6)


,user_id,signup_date,region,subscription_tier,monthly_usage_hours,monthly_bill
0,UID_10000,2024-01-01,EMEA,Pro,32.3,33.9
1,UID_10001,2024-01-01,APAC,Basic,28.2,10.32
2,UID_10002,2024-01-02,APAC,Enterprise,284.9,311.52
3,UID_10003,2024-01-02,North America,Basic,32.4,14.87
4,UID_10004,2024-01-03,EMEA,Pro,68.5,30.99


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   user_id              300 non-null    str  
 1   signup_date          300 non-null    str  
 2   region               300 non-null    str  
 3   subscription_tier    300 non-null    str  
 4   monthly_usage_hours  298 non-null    str  
 5   monthly_bill         299 non-null    str  
dtypes: str(6)
memory usage: 26.0 KB


In [6]:
print("Missing values per column (standard NaN only):")
print(df.isnull().sum())

Missing values per column (standard NaN only):
user_id                0
signup_date            0
region                 0
subscription_tier      0
monthly_usage_hours    2
monthly_bill           1
dtype: int64


**Identifying non-standard missing value flags**

`df.isnull()` only catches *actual* NaNs but this dataset also uses text placeholders like
`"N/A"`, `"missing"`, and `"-999"` to represent missing data, which pandas currently reads as
ordinary strings. Scanning each column's unique values surfaces them.

In [7]:
non_standard_flags = ["N/A", "missing", "-999"]

for col in df.columns:
    found = [v for v in df[col].dropna().unique() if str(v) in non_standard_flags]
    if found:
        print(f"{col}: {found}")

monthly_usage_hours: ['missing', '-999']
monthly_bill: ['missing']


## Task 2: Data Cleaning & Quality Enforcement

**1. Standardize non-standard null values into `np.nan`**

In [8]:
df = df.replace(non_standard_flags, np.nan)

print("Missing values after standardizing null flags:")
print(df.isnull().sum())

Missing values after standardizing null flags:
user_id                0
signup_date            0
region                 0
subscription_tier      0
monthly_usage_hours    4
monthly_bill           2
dtype: int64


**2. Fix data types**

In [9]:
# signup_date proper datetime64
df["signup_date"] = pd.to_datetime(df["signup_date"])

# monthly_usage_hours and monthly_bill were read as text because of the
# "N/A" / "missing" / "-999" placeholders mixed in convert them to numeric now
df["monthly_usage_hours"] = pd.to_numeric(df["monthly_usage_hours"], errors="coerce")
df["monthly_bill"] = pd.to_numeric(df["monthly_bill"], errors="coerce")

df.dtypes

user_id                           str
signup_date            datetime64[us]
region                            str
subscription_tier                 str
monthly_usage_hours           float64
monthly_bill                  float64
dtype: object

**3. Handle data anomalies — negative usage hours**

In [10]:
negative_count = (df["monthly_usage_hours"] < 0).sum()
print("Negative usage-hour readings found:", negative_count)

# A negative usage value is physically impossible treat it as missing, not real data
df.loc[df["monthly_usage_hours"] < 0, "monthly_usage_hours"] = np.nan

print("Total missing monthly_usage_hours now (nulls + former negatives):",
      df["monthly_usage_hours"].isna().sum())

Negative usage-hour readings found: 2
Total missing monthly_usage_hours now (nulls + former negatives): 6


**4. Impute missing `monthly_usage_hours` using each user's subscription-tier median**

In [8]:
df["monthly_usage_hours"] = df["monthly_usage_hours"].fillna(
    df.groupby("subscription_tier")["monthly_usage_hours"].transform("median")
)

print("Missing monthly_usage_hours after imputation:", df["monthly_usage_hours"].isna().sum())

Missing monthly_usage_hours after imputation: 0


*Note: `monthly_bill` still has a couple of missing values at this point. The assignment's
imputation instructions (Task 2, step 4) apply specifically to `monthly_usage_hours`, so
`monthly_bill`'s missing entries are left as `NaN` they're simply excluded automatically by
pandas' aggregation functions (`mean`, `sum`, etc.) later on.*

## Task 3: Feature Engineering with Group Transformations

In [11]:
# tier_avg_usage: average usage hours for each user's subscription tier
df["tier_avg_usage"] = df.groupby("subscription_tier")["monthly_usage_hours"].transform("mean")

# usage_deviation: how far each user sits from their tier's average
# positive -> power user, negative under-utilizer
df["usage_deviation"] = df["monthly_usage_hours"] - df["tier_avg_usage"]

df[["user_id", "subscription_tier", "monthly_usage_hours", "tier_avg_usage", "usage_deviation"]].head()

,user_id,subscription_tier,monthly_usage_hours,tier_avg_usage,usage_deviation
0,UID_10000,Pro,32.3,73.736111,-41.436111
1,UID_10001,Basic,28.2,22.940845,5.259155
2,UID_10002,Enterprise,284.9,232.640909,52.259091
3,UID_10003,Basic,32.4,22.940845,9.459155
4,UID_10004,Pro,68.5,73.736111,-5.236111


## Task 4: Executive Aggregations (Split-Apply-Combine)

In [12]:
executive_summary = df.groupby(["subscription_tier", "region"]).agg(
    total_users=("user_id", "nunique"),
    avg_monthly_bill=("monthly_bill", "mean"),
    total_usage_hours=("monthly_usage_hours", "sum"),
    max_usage_deviation=("usage_deviation", "max")
).reset_index()

executive_summary.round(2)

,subscription_tier,region,total_users,avg_monthly_bill,total_usage_hours,max_usage_deviation
0,Basic,APAC,32,12.35,640.5,16.26
1,Basic,EMEA,37,12.68,836.7,16.96
2,Basic,LATAM,44,12.57,989.9,16.16
3,Basic,North America,33,12.92,790.5,16.46
4,Enterprise,APAC,11,390.78,2509.3,86.86
5,Enterprise,EMEA,12,315.53,3104.3,109.96
6,Enterprise,LATAM,11,345.00,2385.2,84.36
7,Enterprise,North America,12,405.70,2237.4,105.56
8,Pro,APAC,19,39.59,1327.7,35.46
9,Pro,EMEA,28,38.62,1840.1,25.36


## Task 5: Reshaping for Business Reporting

In [13]:
billing_pivot = pd.pivot_table(
    df,
    index="subscription_tier",
    columns="region",
    values="monthly_bill",
    aggfunc="mean",
    margins=True,
    margins_name="All"
)

billing_pivot.round(2)

region,APAC,EMEA,LATAM,North America,All
subscription_tier,,,,,
Basic,12.35,12.68,12.57,12.92,12.63
Enterprise,390.78,315.53,345.00,405.70,363.58
Pro,39.59,38.62,41.22,40.77,40.10
All,87.84,69.31,64.61,78.06,74.40
